# ClarifyMed: FracAtlas X-ray training
Run each cell in order. First choose **Runtime > Change runtime type > GPU** (T4 if available).
Your FracAtlas ZIP should be in **My Drive/ClarifyMed**. This notebook includes the training program; no project upload is required.

This trains an image-level fracture research baseline, not a general abnormality or diagnostic model.
Dataset: [Abedeen et al., Scientific Data (2023)](https://www.nature.com/articles/s41597-023-02432-4),
[FracAtlas on Figshare](https://doi.org/10.6084/m9.figshare.22363012), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
Patient separation cannot be verified without patient IDs. Identical decoded images are kept together; near-duplicates may remain.
Do not tune using test results. Model scores and Grad-CAM require further evaluation before deployment.


## 1. Connect Drive and check GPU
Approve the Google Drive connection when prompted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU connected. Select Runtime > Change runtime type > GPU, reconnect, then run this cell again.')
print('GPU:', torch.cuda.get_device_name(0))


## 2. Install supporting packages
Keep Colab's existing matched PyTorch/torchvision installation.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn>=1.5,<2', 'Pillow>=11,<13'], check=True)
import torchvision
print('PyTorch:', torch.__version__, 'torchvision:', torchvision.__version__)


## 3. Find and extract the ZIP
If multiple ZIPs are found, enter the number of the FracAtlas ZIP. If none are found, paste its full Drive path when prompted. Extraction happens on Colab storage for faster training.

In [ ]:
from pathlib import Path
import shutil, zipfile, tempfile, stat
drive_folder = Path('/content/drive/MyDrive/ClarifyMed')
candidates = sorted(p for p in drive_folder.rglob('*') if p.is_file() and p.suffix.lower() == '.zip') if drive_folder.exists() else []
if len(candidates) == 1:
    zip_path = candidates[0]
elif candidates:
    for index, path in enumerate(candidates, 1):
        print(index, path)
    choice = int(input('Enter the FracAtlas ZIP number: '))
    if not 1 <= choice <= len(candidates):
        raise ValueError('Invalid selection. Rerun this cell.')
    zip_path = candidates[choice - 1]
else:
    zip_path = Path(input('Paste full ZIP path, for example /content/drive/MyDrive/FracAtlas.zip: ').strip())
if not zip_path.is_file() or not zipfile.is_zipfile(zip_path):
    raise ValueError('The selected path is not a valid ZIP. Check that the Drive upload has finished.')
extracted = Path(tempfile.mkdtemp(prefix='clarifymed-fracatlas-', dir='/content'))
local_zip = extracted / 'source.zip'
shutil.copy2(zip_path, local_zip)
with zipfile.ZipFile(local_zip) as archive:
    if sum(info.file_size for info in archive.infolist()) > shutil.disk_usage(extracted).free:
        raise RuntimeError('Not enough Colab disk space to extract this ZIP.')
    for info in archive.infolist():
        target = (extracted / info.filename.replace('\\', '/')).resolve()
        if not target.is_relative_to(extracted.resolve()) or stat.S_ISLNK(info.external_attr >> 16):
            raise ValueError('ZIP contains an unsafe path or symbolic link.')
    archive.extractall(extracted)
roots = [p.parent for p in extracted.rglob('dataset.csv') if (p.parent / 'images').is_dir()]
if len(roots) != 1:
    raise ValueError('Expected one dataset.csv with an images folder beside it. Check that you uploaded the complete FracAtlas ZIP.')
DATA_ROOT = roots[0]
print('Dataset ready:', DATA_ROOT)


## 4. Prepare the training program
This cell writes the included trainer into the temporary Colab runtime.

In [ ]:
trainer_source = '"""FracAtlas image-level research baseline; not a clinically validated model."""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport random\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\nfrom sklearn.model_selection import train_test_split\n\nLABELS = [\'non_fractured\', \'fractured\']\nSCOPE = \'FracAtlas musculoskeletal fracture classification only\'\nLIMITATIONS = (\'Research prototype. Splits isolate identical decoded images, not patients: \'\n               \'patient identifiers are unavailable in dataset.csv. Near-duplicate and patient \'\n               \'leakage remain possible. Non-fractured does not mean normal. \'\n               \'No external clinical validation; Grad-CAM is not confirmed fracture localization.\')\n\n\ndef prepare_dataset(root: Path, seed: int = 42, skip_unreadable: bool = False, exclusions: list | None = None):\n    exclusions = exclusions if exclusions is not None else []\n    root = root.resolve()\n    with (root / \'dataset.csv\').open(encoding=\'utf-8-sig\', newline=\'\') as source:\n        reader = csv.DictReader(source)\n        if not {\'image_id\', \'fractured\'} <= set(reader.fieldnames or []):\n            raise ValueError(\'dataset.csv needs image_id and fractured columns.\')\n        rows = list(reader)\n    if not rows:\n        raise ValueError(\'Dataset is empty.\')\n    images = defaultdict(list)\n    for path in (root / \'images\').rglob(\'*\'):\n        if path.is_file():\n            images[path.name].append(path)\n    seen, groups = set(), defaultdict(list)\n    for index, row in enumerate(rows):\n        name = row[\'image_id\']\n        if name in seen or not images[name]:\n            raise ValueError(f\'Duplicate metadata, missing or ambiguous image: {name}\')\n        seen.add(name)\n        if row[\'fractured\'] not in (\'0\', \'1\'):\n            raise ValueError(f\'Invalid fractured label: {name}\')\n        candidates = sorted(images[name])\n        if len(candidates) > 1:\n            hashes = {hashlib.sha256(candidate.read_bytes()).hexdigest() for candidate in candidates}\n            if len(hashes) != 1:\n                raise ValueError(f\'Ambiguous different images share a filename: {name}\')\n            print(f\'Using CSV label for {name}; {len(candidates)} identical file copies counted once.\', flush=True)\n        path = candidates[0].resolve()\n        if not path.is_relative_to(root):\n            raise ValueError(\'Image path escapes dataset root.\')\n        try:\n            with Image.open(path) as image:\n                rgb = image.convert(\'RGB\')\n                rgb.load()\n                digest = hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()\n        except OSError as error:\n            exclusions.append({\'image_id\': name, \'label\': int(row[\'fractured\']), \'reason\': str(error)})\n            if skip_unreadable:\n                print(f\'Excluded unreadable image: {name}: {error}\', flush=True)\n                continue\n            raise ValueError(f\'Unreadable or truncated image {name}: {error}. Re-extract the original download or obtain a clean copy; do not enable truncated-image loading.\') from error\n        record = {\'image_id\': name, \'path\': path.relative_to(root).as_posix(),\n                  \'label\': int(row[\'fractured\']), \'pixel_sha256\': digest, \'identical_file_copies\': len(candidates),\n                  \'body_parts\': [part for part in (\'hand\', \'leg\', \'hip\', \'shoulder\') if row.get(part) == \'1\']}\n        groups[digest].append(record)\n        if (index + 1) % 500 == 0:\n            print(f\'Validated {index + 1}/{len(rows)} images\', flush=True)\n    if len(exclusions) / len(rows) > .01:\n        raise ValueError(f\'{len(exclusions)}/{len(rows)} images are unreadable (over 1%). Obtain a clean dataset ZIP before training. See data_validation.json.\')\n    if any(len({r[\'label\'] for r in group}) != 1 for group in groups.values()):\n        raise ValueError(\'Identical images have conflicting labels.\')\n    keys = sorted(groups)\n    labels = [groups[key][0][\'label\'] for key in keys]\n    train, heldout = train_test_split(keys, test_size=.30, stratify=labels, random_state=seed)\n    dev, test = train_test_split(heldout, test_size=.50,\n                               stratify=[groups[key][0][\'label\'] for key in heldout], random_state=seed)\n    splits = {name: [row for key in selected for row in groups[key]]\n              for name, selected in [(\'train\', train), (\'dev\', dev), (\'test\', test)]}\n    for name, records in splits.items():\n        if {row[\'label\'] for row in records} != {0, 1}:\n            raise ValueError(f\'{name} needs both classes.\')\n    return splits\n\n\ndef metrics(truth, probabilities):\n    from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score\n    truth, probabilities = np.asarray(truth), np.asarray(probabilities)\n    predicted = (probabilities >= .5).astype(int)\n    confidence = np.maximum(probabilities, 1 - probabilities)\n    error = 0.\n    for index in range(10):\n        mask = (confidence >= index / 10) & (confidence < (index + 1) / 10 if index < 9 else confidence <= 1)\n        if mask.any():\n            error += float(mask.mean() * abs((predicted[mask] == truth[mask]).mean() - confidence[mask].mean()))\n    return {\'count\': len(truth), \'roc_auc\': float(roc_auc_score(truth, probabilities)) if len(set(truth)) == 2 else None,\n            \'average_precision\': float(average_precision_score(truth, probabilities)) if len(set(truth)) == 2 else None,\n            \'classification_report\': classification_report(truth, predicted, labels=[0, 1], target_names=LABELS, output_dict=True, zero_division=0),\n            \'confusion_matrix\': confusion_matrix(truth, predicted, labels=[0, 1]).tolist(),\n            \'expected_calibration_error\': error, \'brier_score\': float(np.mean((probabilities - truth) ** 2))}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--data-root\', required=True, type=Path)\n    parser.add_argument(\'--output-dir\', required=True, type=Path)\n    parser.add_argument(\'--epochs\', type=int, default=8)\n    parser.add_argument(\'--batch-size\', type=int, default=32)\n    parser.add_argument(\'--seed\', type=int, default=42)\n    parser.add_argument(\'--validate-only\', action=\'store_true\')\n    parser.add_argument(\'--skip-unreadable\', action=\'store_true\', help=\'Record and exclude unreadable files, stopping if more than 1 percent are unreadable.\')\n    args = parser.parse_args()\n    if args.epochs < 1 or args.batch_size < 1:\n        raise ValueError(\'epochs and batch-size must be positive.\')\n    if args.output_dir.exists():\n        raise FileExistsError(\'Use a new output directory to preserve previous runs.\')\n    args.output_dir.mkdir(parents=True)\n    exclusions = []\n    try:\n        splits = prepare_dataset(args.data_root, args.seed, args.skip_unreadable, exclusions)\n    except (ValueError, OSError) as error:\n        (args.output_dir / \'data_validation.json\').write_text(json.dumps({\'valid\': False, \'error\': str(error), \'excluded_images\': exclusions}, indent=2), encoding=\'utf-8\')\n        raise\n    (args.output_dir / \'data_validation.json\').write_text(json.dumps({\'valid\': True, \'excluded_images\': exclusions}, indent=2), encoding=\'utf-8\')\n    manifest = {\'seed\': args.seed, \'split_method\': \'70/15/15 stratified decoded-image-hash groups; custom classification split\',\n                \'limitations\': LIMITATIONS, \'dataset_csv_sha256\': hashlib.sha256((args.data_root / \'dataset.csv\').read_bytes()).hexdigest(),\n                \'excluded_images\': exclusions, \'counts\': {name: dict(Counter(LABELS[r[\'label\']] for r in rows)) for name, rows in splits.items()}, \'splits\': splits}\n    manifest_path = args.output_dir / \'split_manifest.json\'\n    manifest_path.write_text(json.dumps(manifest, indent=2), encoding=\'utf-8\')\n    print(json.dumps(manifest[\'counts\'], indent=2), flush=True)\n    if args.validate_only:\n        print(\'Dataset validation complete. No model trained.\', flush=True)\n        return\n\n    import torch\n    import torchvision\n    import sklearn\n    from torch import nn\n    from torch.utils.data import Dataset, DataLoader\n    from torchvision import models, transforms\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Select a GPU in Colab: Runtime > Change runtime type, then rerun with a new output directory.\')\n    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed); torch.cuda.manual_seed_all(args.seed)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n    normalize = transforms.Normalize([.485, .456, .406], [.229, .224, .225])\n    evaluation_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), normalize])\n    training_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.RandomRotation(7), transforms.ToTensor(), normalize])\n\n    class Images(Dataset):\n        def __init__(self, rows, transform):\n            self.rows, self.transform = rows, transform\n        def __len__(self):\n            return len(self.rows)\n        def __getitem__(self, index):\n            row = self.rows[index]\n            with Image.open(args.data_root / row[\'path\']) as image:\n                tensor = self.transform(image.convert(\'RGB\'))\n            return tensor, row[\'label\']\n\n    loaders = {name: DataLoader(Images(rows, training_transform if name == \'train\' else evaluation_transform),\n                batch_size=args.batch_size, shuffle=name == \'train\', num_workers=0, pin_memory=True)\n               for name, rows in splits.items()}\n    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)\n    model.classifier[1] = nn.Linear(model.last_channel, 2)\n    model.cuda()\n    counts = Counter(row[\'label\'] for row in splits[\'train\'])\n    class_weights = torch.tensor([len(splits[\'train\']) / (2 * counts[i]) for i in range(2)], device=\'cuda\')\n    loss_fn = nn.CrossEntropyLoss(weight=class_weights)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)\n    best_loss, history = float(\'inf\'), []\n    checkpoint_path = args.output_dir / \'model.pt\'\n    for epoch in range(args.epochs):\n        losses = {}\n        for phase in [\'train\', \'dev\']:\n            model.train(phase == \'train\')\n            total, denominator = 0., 0.\n            for batch_index, (images, labels) in enumerate(loaders[phase]):\n                images, labels = images.cuda(), labels.cuda()\n                with torch.set_grad_enabled(phase == \'train\'):\n                    if phase == \'train\':\n                        optimizer.zero_grad(set_to_none=True)\n                    loss = loss_fn(model(images), labels)\n                    if phase == \'train\':\n                        loss.backward(); optimizer.step()\n                weight_sum = class_weights[labels].sum().item()\n                total += loss.item() * weight_sum\n                denominator += weight_sum\n                if phase == \'train\' and (batch_index + 1) % 20 == 0:\n                    print(f\'Epoch {epoch + 1}/{args.epochs}: batch {batch_index + 1}/{len(loaders[phase])}\', flush=True)\n            losses[phase] = total / denominator\n        history.append({\'epoch\': epoch + 1, **losses})\n        if losses[\'dev\'] < best_loss:\n            best_loss = losses[\'dev\']\n            checkpoint = {\'state_dict\': {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},\n                          \'architecture\': \'mobilenet_v2\', \'dataset\': \'FracAtlas\', \'labels\': LABELS, \'scope\': SCOPE,\n                          \'image_size\': 224, \'normalization_mean\': [.485, .456, .406], \'normalization_std\': [.229, .224, .225],\n                          \'threshold\': .5, \'best_epoch\': epoch + 1, \'seed\': args.seed, \'clinically_validated\': False}\n            temporary = args.output_dir / \'model.tmp\'\n            torch.save(checkpoint, temporary)\n            temporary.replace(checkpoint_path)\n        (args.output_dir / \'training_history.json\').write_text(json.dumps(history, indent=2), encoding=\'utf-8\')\n        print(f\'Epoch {epoch + 1}: train={losses["train"]:.4f}, dev={losses["dev"]:.4f}; best checkpoint saved\', flush=True)\n\n    checkpoint = torch.load(checkpoint_path, map_location=\'cpu\', weights_only=True)\n    model.load_state_dict(checkpoint[\'state_dict\']); model.eval()\n    truth, probabilities = [], []\n    with torch.inference_mode():\n        for images, labels in loaders[\'test\']:\n            probabilities.extend(torch.softmax(model(images.cuda()), dim=1)[:, 1].cpu().tolist())\n            truth.extend(labels.tolist())\n    result = {\'dataset\': \'FracAtlas\', \'labels\': LABELS, \'threshold\': .5, \'best_epoch\': checkpoint[\'best_epoch\'],\n              \'epochs_requested\': args.epochs, \'seed\': args.seed, \'test_metrics\': metrics(truth, probabilities),\n              \'split_counts\': manifest[\'counts\'], \'limitations\': LIMITATIONS,\n              \'versions\': {\'torch\': str(torch.__version__), \'torchvision\': str(torchvision.__version__), \'scikit_learn\': sklearn.__version__, \'numpy\': np.__version__},\n              \'model_sha256\': hashlib.sha256(checkpoint_path.read_bytes()).hexdigest(),\n              \'manifest_sha256\': hashlib.sha256(manifest_path.read_bytes()).hexdigest()}\n    result[\'test_by_body_part\'] = {}\n    for part in (\'hand\', \'leg\', \'hip\', \'shoulder\'):\n        selected = [i for i, row in enumerate(splits[\'test\']) if part in row[\'body_parts\']]\n        if selected:\n            result[\'test_by_body_part\'][part] = metrics([truth[i] for i in selected], [probabilities[i] for i in selected])\n    (args.output_dir / \'evaluation.json\').write_text(json.dumps(result, indent=2), encoding=\'utf-8\')\n    predictions = [{\'image_id\': row[\'image_id\'], \'label\': label, \'fracture_score\': score}\n                   for row, label, score in zip(splits[\'test\'], truth, probabilities)]\n    (args.output_dir / \'test_predictions.json\').write_text(json.dumps(predictions, indent=2), encoding=\'utf-8\')\n    (args.output_dir / \'MODEL_CARD.md\').write_text(\'# ClarifyMed FracAtlas MobileNetV2\\n\\n\' + LIMITATIONS +\n        \'\\n\\nImageNet transfer learning; weighted cross-entropy from training counts; checkpoint selected by development loss. Fixed threshold 0.5. \'\n        \'Custom classification splits differ from the supplied fracture-only localization splits. See evaluation.json for actual results. \'\n        \'Scores are uncalibrated model outputs.\\n\\nDataset attribution: Abedeen et al., Scientific Data 10, 521 (2023). \'\n        \'https://www.nature.com/articles/s41597-023-02432-4 ; dataset https://doi.org/10.6084/m9.figshare.22363012 ; \'\n        \'CC BY 4.0 https://creativecommons.org/licenses/by/4.0/ . Images resized and augmented for training.\\n\', encoding=\'utf-8\')\n    print(json.dumps(result[\'test_metrics\'], indent=2), flush=True)\n    print(f\'Training complete. Artifacts: {args.output_dir}\', flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n'
Path('/content/train_fracatlas.py').write_text(trainer_source, encoding='utf-8')
print('Training program ready.')


## 5. Validate images and train
Start with the defaults: 8 epochs, batch size 32. Each run gets a new Drive output folder.
This cell checks images, makes reproducible splits, trains MobileNetV2, saves the best development checkpoint, then evaluates once on test data.
Unreadable images are recorded in data_validation.json and excluded (at most 1%); larger damage stops the run and requires a clean ZIP.
Expect image-check messages followed by epoch progress. Keep Colab open while it runs.
If the session ends early, the latest best checkpoint may be saved, but the run is **incomplete** without evaluation.json.
Automatic resume is not implemented; a rerun starts a new experiment and preserves previous results.


In [ ]:
from datetime import datetime, timezone
import uuid
EPOCHS = 8
BATCH_SIZE = 32
run_name = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S') + '-' + uuid.uuid4().hex[:6]
OUTPUT_DIR = drive_folder / 'models' / ('fracatlas-' + run_name)
print('Results will be saved to:', OUTPUT_DIR, flush=True)
subprocess.run([sys.executable, '-u', '/content/train_fracatlas.py',
                '--data-root', str(DATA_ROOT), '--output-dir', str(OUTPUT_DIR),
                '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE), '--skip-unreadable'], check=True)


## 6. View results and create an artifact ZIP
After training, share evaluation.json for review. The website supports FracAtlas checkpoints through XRAY_MODEL_PATH after evaluation; do not install these weights as a MURA model.

In [ ]:
import json
evaluation_file = OUTPUT_DIR / 'evaluation.json'
if not evaluation_file.is_file():
    raise RuntimeError('Training/evaluation has not completed. Inspect the preceding cell error.')
evaluation = json.loads(evaluation_file.read_text())
print(json.dumps(evaluation['test_metrics'], indent=2))
archive_path = shutil.make_archive(str(OUTPUT_DIR), 'zip', root_dir=OUTPUT_DIR)
print('Saved model, evaluation, split manifest, predictions, and model card to:', OUTPUT_DIR)
print('Artifact ZIP:', archive_path)
print('Next: share evaluation.json with Codex for review and app integration.')
